<a href="https://colab.research.google.com/github/matizzat/NanoMed/blob/main/cross_validation/predictions_random_forest_ja.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import pandas as pd
import joblib
import io
import os
import zipfile
import gdown
from google.colab import files

# =============================================================================
# 1. CONFIGURACIÓN (Pon aquí el ID de tu ZIP de Google Drive)
# =============================================================================
FILE_ID_ZIP = '1Zpf9tVSZi7mQXoMhq5MwnsmZ7nm5UZ9p'  # <--- ¡IMPORTANTE! Reemplaza esto
CARPETA_MODELOS = './modelos_proteina'

# Variables globales para listas (Defínelas aquí para que todo funcione)
selected_attributes = [
    'np_without_modification', 'surface_modification', 'zeta_potential',
    'incubation_protein_source', 'incubation_plasma_concentration', 'incubation_np_concentration',
    'np_type', 'np_shape', 'dispersion_medium', 'dispersion_medium_ph', 'size_dls', 'pdi',
    'incubation_culture', 'incubation_time', 'incubation_temperature', 'modification_type'
]

categorical_attributes = [
    'np_without_modification', 'surface_modification', 'incubation_protein_source',
    'np_type', 'np_shape', 'dispersion_medium', 'incubation_culture', 'modification_type'
]

protein_targets = [
    "P01871", "P01024", "P02647", "P02649", "P02768", "P04004", "P01834",
    "P10909", "P02652", "P00734", "P01009", "P01042", "P01857", "P01859",
    "P01023", "P01619", "P04196", "P02656", "P04114", "P02787", "P02766",
    "P06396", "P06727", "P0C0L5", "P02671", "P08603", "P02765", "Q14624",
    "P0DOY2", "P01008", "P68871", "P01764", "P04003", "P0C0L4", "P01876",
    "P02749", "P02675", "P01860", "P01011", "P00747", "P02774", "P00751",
    "P19823", "P08697", "P19827", "P69905", "P02760", "P02748", "P02679",
    "P02790", "B9A064", "P07996", "P05155", "P01766", "P12259", "P00738",
    "P02751", "P02654", "P04406", "P00739", "P02655", "P00736", "P07225",
    "P05154", "P09871", "P60709", "P03952", "P02747", "P05546", "P02746",
    "P27169", "P01019", "P35542", "P04217", "Q14520", "P02743", "P02763",
    "P49908", "O43866", "P18428", "P55056", "P04264", "P00748", "P01591",
    "P01861", "Q92954", "P02776", "P0DJI8", "P01031", "P05156", "P13671",
    "Q03591", "P06312", "P00740", "P01615", "P00742", "P04070", "O14791",
    "P05090", "P02745", "P00488", "Q13103", "P01877", "P01700", "P10643",
    "Q9UK55", "P03951", "Q06033", "Q96IY4", "P13645", "P04433", "P07358",
    "P27918", "P05452", "P20851", "P07357", "P07360", "P01599", "O95445",
    "Q9BXR6", "P02775", "P02741", "P35579", "P15169", "Q13790", "P35858",
    "P49747", "P36955", "P19652", "P07737", "P22891", "P35908", "P80748",
    "P08514", "P01593", "Q04756", "P06733", "P23528", "P63104", "P18065",
    "P08519", "Q86UX7", "P02753", "Q5TB80", "P22352", "P00746", "P35443",
    "P62937", "P10720", "P48740", "P25311", "P43652", "P35527", "Q9Y490",
    "P05106", "P17936", "P18206", "P02788", "P06702", "P01701", "Q6Q788",
    "Q96PD5", "Q9UGM5", "O00391", "P11142", "P14618", "P23142", "P68366",
    "P12814", "P61224", "Q13201", "P67936", "P0DOY3", "P08709", "P01034",
    "P81605", "P02533", "P11226"]

# =============================================================================
# 2. FUNCIONES DE UTILIDAD Y SETUP
# =============================================================================

def setup_environment(file_id, output_folder):
    """Descarga modelos desde Drive público y los descomprime."""
    if os.path.exists(output_folder) and len(os.listdir(output_folder)) > 0:
        return True

    print("⬇️ Descargando modelos...")
    output_zip = 'modelos_temp.zip'
    url = f'https://drive.google.com/uc?id={file_id}'

    try:
        gdown.download(url, output_zip, quiet=False)
        print("📦 Descomprimiendo...")
        with zipfile.ZipFile(output_zip, 'r') as zip_ref:
            zip_ref.extractall(output_folder)

        # Mover archivos si quedaron en subcarpetas
        for root, dirs, files_list in os.walk(output_folder):
            for file in files_list:
                if file.endswith('.joblib') and root != output_folder:
                    import shutil
                    shutil.move(os.path.join(root, file), os.path.join(output_folder, file))
        return True
    except Exception as e:
        print(f"❌ Error en setup: {e}")
        return False

def generar_y_guardar_columnas(df_train, categorical_attributes, selected_attributes, output_path):
    """Genera la lista de columnas maestra basada en el CSV de entrenamiento."""
    try:
        X = df_train[selected_attributes]
        X = pd.get_dummies(X, columns=categorical_attributes)

        # Guardamos la lista de columnas
        cols_path = os.path.join(output_path, 'columnas_entrenamiento.joblib')
        joblib.dump(X.columns, cols_path)
        print(f"✅ Estructura guardada exitosamente en: {cols_path}")
        return X.columns
    except Exception as e:
        print(f"❌ Error al generar columnas: {e}")
        return None

# =============================================================================
# 3. FUNCIÓN DE PREDICCIÓN CORREGIDA
# =============================================================================

def predict_csv_gui(df_input, models_path, selected_attributes, categorical_attributes, protein_targets):
    """
    Versión corregida para la GUI.
    1. Acepta DataFrame directo.
    2. Carga la estructura de columnas desde archivo.
    3. Maneja nuevas clases o clases faltantes con reindex.
    """

    # 1. Cargar la estructura de columnas requerida
    cols_file = os.path.join(models_path, 'columnas_entrenamiento.joblib')
    if not os.path.exists(cols_file):
        return "❌ Error: No se encontró 'columnas_entrenamiento.joblib'. Debes subir el CSV de entrenamiento primero (Paso 2 de la interfaz)."

    try:
        train_columns = joblib.load(cols_file)
    except:
        return "❌ Error leyendo el archivo de columnas."

    # 2. Preprocesamiento
    # Trabajamos sobre una copia
    df_work = df_input.copy()

    # Verificar que existan los atributos base
    missing = [col for col in selected_attributes if col not in df_work.columns]
    if missing:
        return f"❌ El archivo subido no tiene las columnas: {missing}"

    X_test = df_work[selected_attributes]

    # One-Hot Encoding (Aquí aparecen las clases nuevas que te preocupan)
    X_test = pd.get_dummies(X_test, columns=categorical_attributes)

    # 3. ALINEACIÓN MÁGICA (Solución a tu duda)
    # train_columns: Son las columnas que el modelo CONOCE.
    # Si X_test tiene una columna nueva (ej: 'Color_Magenta') que no está en train_columns, se borra.
    # Si X_test le falta una columna (ej: 'Color_Rojo'), se agrega con valor 0.
    X_test = X_test.reindex(columns=train_columns, fill_value=0)

    # 4. Predicción
    predictions_count = 0
    for target in protein_targets:
        model_file = os.path.join(models_path, f'random_forest_regressor_model_{target}.joblib')

        if os.path.exists(model_file):
            try:
                model = joblib.load(model_file)
                prediction = model.predict(X_test)
                df_input[target + '_prediccion'] = prediction
                predictions_count += 1
            except Exception as e:
                print(f"⚠️ Error prediciendo {target}: {e}")
        # else:
            # print(f"⚠️ Modelo no encontrado para {target}")

    if predictions_count == 0:
        return "⚠️ No se pudieron realizar predicciones (no se encontraron modelos válidos)."

    return df_input

# =============================================================================
# 4. INTERFAZ GRÁFICA
# =============================================================================

style = {'description_width': 'initial'}
header = widgets.HTML("<h2>🧬 App de Predicción de Corona Proteica</h2>")

# --- Paso 1: Inicializar ---
btn_init = widgets.Button(description='1. Descargar Modelos', button_style='info', icon='cloud-download', layout=widgets.Layout(width='100%'))

# --- Paso 2: Configurar Columnas (Solo si es necesario) ---
lbl_train = widgets.HTML("<b>2. Configuración (Solo la primera vez o si faltan columnas):</b><br>Sube el archivo 'individual_proteins_dataset.csv'")
upl_train = widgets.FileUpload(accept='.csv', description='Subir Entrenamiento', multiple=False)
btn_train = widgets.Button(description='Generar Estructura', button_style='warning')

# --- Paso 3: Predecir ---
lbl_pred = widgets.HTML("<b>3. Predicción:</b><br>Sube el archivo con los nuevos datos.")
upl_pred = widgets.FileUpload(accept='.csv', description='Subir Nuevos Datos', multiple=False)
btn_pred = widgets.Button(description='Ejecutar Predicción', button_style='success', icon='play', layout=widgets.Layout(width='100%'))

out_log = widgets.Output()

def on_init_click(b):
    with out_log:
        clear_output()
        if setup_environment(FILE_ID_ZIP, CARPETA_MODELOS):
            print("✅ Modelos descargados y listos.")
        else:
            print("❌ Error en la descarga. Verifica el ID.")

def on_train_click(b):
    with out_log:
        if not upl_train.value:
            print("⚠️ Sube el archivo de entrenamiento primero.")
            return

        # Leer CSV subido
        if isinstance(upl_train.value, tuple): f = upl_train.value[0]
        else: key = list(upl_train.value.keys())[0]; f = upl_train.value[key]

        df_train = pd.read_csv(io.BytesIO(f['content']))
        generar_y_guardar_columnas(df_train, categorical_attributes, selected_attributes, CARPETA_MODELOS)

def on_pred_click(b):
    with out_log:
        if not upl_pred.value:
            print("⚠️ Sube el archivo de nuevos datos primero.")
            return

        # Leer CSV subido
        if isinstance(upl_pred.value, tuple): f = upl_pred.value[0]
        else: key = list(upl_pred.value.keys())[0]; f = upl_pred.value[key]

        print("⏳ Procesando...")
        df_new = pd.read_csv(io.BytesIO(f['content']))

        # LLAMADA A LA FUNCIÓN CORREGIDA
        resultado = predict_csv_gui(
            df_new,
            CARPETA_MODELOS,
            selected_attributes,
            categorical_attributes,
            protein_targets
        )

        if isinstance(resultado, str):
            print(resultado)
        else:
            print("✅ ¡Predicción finalizada! Descargando archivo...")
            resultado.to_csv('predicciones_finales.csv', index=False)
            files.download('predicciones_finales.csv')

btn_init.on_click(on_init_click)
btn_train.on_click(on_train_click)
btn_pred.on_click(on_pred_click)

ui = widgets.VBox([
    header,
    btn_init, widgets.HTML("<hr>"),
    lbl_train, widgets.HBox([upl_train, btn_train]), widgets.HTML("<hr>"),
    lbl_pred, upl_pred, btn_pred,
    out_log
])

display(ui)